# 06a — Tokens, attention, and decoding — Worked answers

Compare reasoning as well as output. Close this file and repeat with changed inputs after 48 hours.

**Prerequisites:** earlier lessons in the course order. **Environment:** Install requirements/core.txt.

[Stage guide](../stages/06.md) · [Course map](../PLAN.md) · [Project](../../projects/stage06/README.md)

**Notebook time:** 2–4 hours for an initial pass; project, reading, retrieval practice, and independent transfer use the rest of the stage budget.

Learning objectives:

- Implement a tokenizer baseline
- Compute masked scaled dot-product attention
- Explain context, sampling, and model families


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets" / "financials.csv").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the learning repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def expect_error(error_type, operation):
    try:
        operation()
    except error_type:
        return
    raise AssertionError(f"Expected {error_type.__name__}")

print("Course root:", ROOT)


## Lesson

A tokenizer maps text to IDs. Character tokenizers are simple but inefficient; subword tokenizers balance vocabulary size and sequence length. Token IDs are indices, not meaningful numerical magnitudes. An embedding table maps IDs to learned vectors. Positional information lets a transformer distinguish order.

Attention computes scores QKᵀ/√d, normalizes each query row with softmax, and combines value vectors. A causal mask prevents attending to future positions. Multiple heads learn different projections; residual connections and normalization stabilize deep stacks. Encoders such as BERT use bidirectional context; decoder families such as GPT and many Llama/Qwen/Mistral models generate autoregressively; T5 uses encoder-decoder structure. Family names do not specify every variant.

Greedy decoding selects the highest score. Temperature rescales logits; top-k restricts to k choices; top-p retains a probability-mass prefix. Beam search tracks multiple sequences. A context window limits tokens the model can consider. A KV cache reuses past attention keys and values, trading memory for less recomputation.


## Worked example

Predict the output before running. Change one input and explain the result.


In [ ]:
import numpy as np
scores = np.array([1., 2., 3.])
p = np.exp(scores-scores.max())
print(p/p.sum())


## Exercise 1: Character tokenizer

Build sorted unique-character vocabulary. Return ids, encode mapping, and decoded text for the given nonempty string.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def tokenize(text):
    vocabulary = sorted(set(text))
    encode = {char: i for i, char in enumerate(vocabulary)}
    ids = [encode[char] for char in text]
    return ids, encode, "".join(vocabulary[i] for i in ids)


In [ ]:
ids, vocab, decoded = tokenize("banana")
assert decoded == "banana" and len(vocab) == 3
print("Exercise 1: checks passed")


**Why this works:** Real tokenizers also need handling for unknown text, normalization, and special tokens.


## Exercise 2: Causal attention

Return output and attention weights for equal-length 2D Q,K,V. Mask all future positions before stable row softmax.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def attention(q, k, v):
    scores = q@k.T/np.sqrt(q.shape[-1])
    scores = np.where(np.triu(np.ones(scores.shape), 1).astype(bool), -np.inf, scores)
    weights = np.exp(scores-scores.max(axis=-1, keepdims=True))
    weights /= weights.sum(axis=-1, keepdims=True)
    return weights@v, weights


In [ ]:
out, weights = attention(np.eye(3), np.eye(3), np.eye(3))
assert np.allclose(weights.sum(axis=1), 1)
assert np.allclose(np.triu(weights, 1), 0)
assert np.allclose(out[0], [1, 0, 0])
print("Exercise 2: checks passed")


**Why this works:** A causal mask is a training-data boundary: future tokens would leak the answer.


## Exercise 3: Temperature distribution

Return stable softmax(logits/temperature). Reject nonpositive temperature.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def distribution(logits, temperature=1.):
    if temperature <= 0: raise ValueError("Temperature must be positive")
    values = np.asarray(logits)/temperature
    p = np.exp(values-values.max())
    return p/p.sum()


In [ ]:
assert distribution([0, 1], .1)[1] > distribution([0, 1], 2)[1]
expect_error(ValueError, lambda: distribution([1, 2], 0))
print("Exercise 3: checks passed")


**Why this works:** Lower temperature sharpens a distribution; it does not guarantee factual correctness.


## Independent transfer

Visualize attention weights and prove changing a future value cannot affect an earlier output. Implement top-k/top-p and compare entropy. Read a subword tokenizer vocabulary and explain special tokens.

Record your implementation, errors, measurements, and explanation in work/. See the project’s ANSWERS.md after attempting the brief.


## Explain without code

1. Why divide by square root of d?

   It moderates dot-product scale as dimension grows, helping avoid saturated softmax.

2. Does attention prove explanation?

   No. Attention weights alone do not establish a causal explanation of a model’s decision.


## Reading and review

- [Primary reference 1](https://huggingface.co/learn/llm-course/chapter1/1)
- [Primary reference 2](https://arxiv.org/abs/1706.03762)

After 2, 7, and 30 days: explain the concept from memory, solve a changed-input version, and log any gaps in progress.md.
